<div style="padding:2.2rem;border-radius:18px;background:linear-gradient(135deg,#7c2d12,#0f766e);color:white;">
<p style="font-size:1.05rem;letter-spacing:.12em;text-transform:uppercase;margin:0;">D151 · MySQL Data Loading</p>
<h1 style="font-size:3rem;margin:.5rem 0;">Import CSV Files into MySQL</h1>
<p style="font-size:1.35rem;margin:0;">Load QuickCart products and orders safely from Windows</p>
</div>

This notebook provides commands to run in **MySQL Workbench** or the **MySQL command-line client**.

# Learning outcomes

By the end, you should be able to:

- Prepare a CSV file for reliable importing
- Create MySQL tables with matching column types
- Import data with `LOAD DATA LOCAL INFILE`
- Use MySQL Workbench's Table Data Import Wizard
- Verify row counts and inspect imported records
- Diagnose common path, permission, and data-format errors

# QuickCart sample files

Two ready-to-use files are included beside this notebook:

| File | Purpose | Rows |
|---|---|---:|
| `products.csv` | Product catalog, prices, and stock | 5 |
| `orders.csv` | Customer orders and totals | 5 |

The first row contains column names. Dates use `YYYY-MM-DD`, and decimal values do not contain currency symbols.

> Open the files in a text editor before loading them. A spreadsheet may silently change dates or identifiers.

# Step 1 — Copy the CSV files to `C:\data`

Create the folder and copy both files using File Explorer, or run this in **PowerShell** from the course workspace:

```powershell
New-Item -ItemType Directory -Path 'C:\data' -Force
Copy-Item '.\D15_DML\products.csv' 'C:\data\products.csv'
Copy-Item '.\D15_DML\orders.csv' 'C:\data\orders.csv'
Get-ChildItem 'C:\data\*.csv'
```

Expected files:

```text
C:\data\products.csv
C:\data\orders.csv
```

# Step 2 — Check CSV structure

`products.csv` begins like this:

```csv
product_id,sku,product_name,category,price,stock_quantity
101,EL-MOU-01,Wireless Mouse,Electronics,799.00,120
```

A clean import requires:

- One header row
- One record per line
- The same number and order of fields on every line
- Commas inside text surrounded by double quotes
- Blank values used consistently for missing data
- UTF-8 encoding where possible

# Step 3 — Create the database

Run in MySQL Workbench or the MySQL client:

```sql
CREATE DATABASE IF NOT EXISTS quickcart_import
  CHARACTER SET utf8mb4
  COLLATE utf8mb4_0900_ai_ci;

USE quickcart_import;
```

The character set allows product and customer text from many languages.

# Step 4 — Create the `products` table

The table columns follow the CSV header order and use suitable types.

```sql
CREATE TABLE IF NOT EXISTS products (
    product_id     INT PRIMARY KEY,
    sku            VARCHAR(30) NOT NULL UNIQUE,
    product_name   VARCHAR(100) NOT NULL,
    category       VARCHAR(80) NOT NULL,
    price          DECIMAL(10, 2) NOT NULL,
    stock_quantity INT NOT NULL
);
```

`DECIMAL` is preferred for money because it stores exact decimal values.

# Step 5 — Create the `orders` table

```sql
CREATE TABLE IF NOT EXISTS orders (
    order_id      INT PRIMARY KEY,
    customer_name VARCHAR(100) NOT NULL,
    customer_email VARCHAR(255) NOT NULL,
    order_date    DATE NOT NULL,
    status        VARCHAR(20) NOT NULL,
    total_amount  DECIMAL(12, 2) NOT NULL
);
```

> This lesson keeps the import example simple. A production system would normally store customers and order items in separate normalized tables.

# Option 1 — Enable local file loading

Check whether the MySQL server accepts local imports:

```sql
SHOW VARIABLES LIKE 'local_infile';
```

If it is `OFF`, an administrator can enable it:

```sql
SET GLOBAL local_infile = 1;
```

Reconnect after changing the global setting. The client must also allow local loading. For the command-line client:

```powershell
mysql --local-infile=1 -u root -p
```

> Enable this only for trusted files and clients.

# Option 1 — Import `products.csv`

Use forward slashes in MySQL file paths on Windows:

```sql
LOAD DATA LOCAL INFILE 'C:/data/products.csv'
INTO TABLE products
CHARACTER SET utf8mb4
FIELDS TERMINATED BY ','
OPTIONALLY ENCLOSED BY '"'
LINES TERMINATED BY '\n'
IGNORE 1 LINES
(product_id, sku, product_name, category, price, stock_quantity);
```

`IGNORE 1 LINES` skips the header. The final list maps CSV fields explicitly to table columns.

# Option 1 — Import `orders.csv`

```sql
LOAD DATA LOCAL INFILE 'C:/data/orders.csv'
INTO TABLE orders
CHARACTER SET utf8mb4
FIELDS TERMINATED BY ','
OPTIONALLY ENCLOSED BY '"'
LINES TERMINATED BY '\n'
IGNORE 1 LINES
(order_id, customer_name, customer_email, order_date, status, total_amount);
```

For a Windows-generated file that does not load correctly, try `LINES TERMINATED BY '\r\n'`. The included sample files use `\n`.

# Verify the import

Never assume that a successful command means the data is correct.

```sql
SELECT COUNT(*) AS product_count FROM products;
SELECT COUNT(*) AS order_count FROM orders;

SELECT * FROM products ORDER BY product_id;
SELECT * FROM orders ORDER BY order_id;

SHOW WARNINGS;
```

Expected counts: **5 products** and **5 orders**. Check dates, decimals, special characters, missing values, and warnings.

# Option 2 — MySQL Workbench Import Wizard

For a visual workflow:

1. Open MySQL Workbench and connect to the server.
2. In **Schemas**, select `quickcart_import`.
3. Right-click the target table and choose **Table Data Import Wizard**.
4. Select `C:\data\products.csv` or `C:\data\orders.csv`.
5. Choose the matching existing table.
6. Confirm that source columns map to the correct table columns.
7. Run the import and review its log.
8. Run the verification queries from the previous slide.

The wizard is convenient for small, occasional loads; `LOAD DATA` is more repeatable and automatable.

# Reloading without duplicate-key errors

Running the import twice attempts to reuse the same primary keys. For this repeatable classroom exercise, clear the tables first:

```sql
USE quickcart_import;
TRUNCATE TABLE orders;
TRUNCATE TABLE products;
```

Then run both `LOAD DATA` statements again.

> `TRUNCATE` removes every row. Do not use it on valuable data. Production pipelines commonly load into a staging table, validate the data, and then merge it into target tables.

# Troubleshooting and recap

| Symptom | Likely cause | Check |
|---|---|---|
| Loading is not allowed | `local_infile` disabled | Server variable and client option |
| File not found | Incorrect path or client cannot read it | Confirm `C:/data/...` and file access |
| Header appears as data | Header was not skipped | Add `IGNORE 1 LINES` |
| Columns are shifted | Wrong delimiter or unquoted commas | Inspect raw CSV text |
| Bad dates or decimals | Source format does not match type | Use ISO dates and plain decimal values |
| Duplicate primary key | File was loaded earlier | Clear the practice table or use a staging workflow |

### Reliable import workflow

**Inspect CSV → create matching table → copy file → import → check warnings → verify rows**